# Tusoai test scenarios

This notebook sets up two optimization tests:
1) **Runner-inline**: target method is inside the runner script (no repo_root).
2) **Repo-root**: runner imports target method from an external repo file.

Hints are set to request writing `runner.txt` (inline case) and `repo.txt` (repo case).

In [ ]:
from pathlib import Path

import openai
from tusoai import Tusoai


In [ ]:
# Configure your API key before running
# import os
# os.environ['OPENAI_API_KEY'] = '...'

client = openai.OpenAI()
tusoai_cfg = Tusoai(
    client=client,
    provider='openai',
    temperature=0.7,
    max_tokens=12000,
    model_settings={
        'pdf': {'model': 'gpt-5.4-nano', 'thinking': False, 'thinking_tokens': 0, 'reasoning_mode': 'low'},
        'construction': {'model': 'gpt-5.4', 'thinking': True, 'thinking_tokens': 2000, 'reasoning_mode': 'medium'},
        'optimization': {'model': 'gpt-5.4-mini', 'thinking': True, 'thinking_tokens': 2000, 'reasoning_mode': 'medium'},
    },
)

In [ ]:
# Shared config
task_description = 'deconvolution of spatial gene expression'
data_available = 'spatial transcriptomics and matched scRNA-seq and code editing the loss function only'
semantic_scholar_api_key = None  # optional


## Scenario 1: runner contains the method (no repo_root)

In [ ]:
cache_dir_1 = 'tusoai_test_runner_inline'
runner_1 = 'testing_scripts/runner_inline/tangram_runner_inline.py'

method_task_1, method_cost_1 = tusoai_cfg.create_method_subtask(
    function_name='_loss_fn',
    task_description=task_description,
    data_available=data_available,
    num_cat=6,
    instruction_count=6,
    num_init=3,
    paper_searches=3,
    info_per_paper=6,
    clear=True,
    cache_dir=cache_dir_1,
    semantic_scholar_api_key=semantic_scholar_api_key,
    hints=[
        'Write a file named runner.txt to indicate this scenario worked.',
        'Keep the function header and output shape unchanged.',
    ],
    use_initial=True,
    source_path=runner_1,
)
method_cost_1

In [ ]:
best_model_1, history_1 = tusoai_cfg.optimize(
    method_tasks=[method_task_1],
    data_tasks=[],
    reference_filename=runner_1,
    timeout=180,
    bug_retries=2,
    n_feedback_buffer=4,
    skip_timeout=True,
    prompt_samples=3,
    drop_island_iter=20,
    prompt_decay=1.5,
    prompt_importance=5.0,
    max_islands=2,
    output_dir=cache_dir_1,
    history_name='runner_inline_test',
    TIME_LIMIT=20,
    task_description=task_description,
    debug=False,
    min_improvement=0.01,
    n_jobs=1,
    COST_LIMIT=5,
)
Path('runner.txt').exists()

## Scenario 2: runner imports method from external repo_root

In [ ]:
cache_dir_2 = 'tusoai_test_repo_case'
runner_2 = 'testing_scripts/repo_case/tangram_runner_repo.py'
repo_root_2 = 'testing_scripts/repo_case'
source_file_2 = 'pkg/mapping_optimizer.py'

method_task_2, method_cost_2 = tusoai_cfg.create_method_subtask(
    function_name='_loss_fn',
    task_description=task_description,
    data_available=data_available,
    num_cat=6,
    instruction_count=6,
    num_init=3,
    paper_searches=3,
    info_per_paper=6,
    clear=True,
    cache_dir=cache_dir_2,
    semantic_scholar_api_key=semantic_scholar_api_key,
    hints=[
        'Write a file named repo.txt to indicate this scenario worked.',
        'Keep the function header and output shape unchanged.',
    ],
    use_initial=True,
    repo_root=repo_root_2,
    source_path=source_file_2,
)
method_cost_2

In [ ]:
best_model_2, history_2 = tusoai_cfg.optimize(
    method_tasks=[method_task_2],
    data_tasks=[],
    reference_filename=runner_2,
    timeout=180,
    bug_retries=2,
    n_feedback_buffer=4,
    skip_timeout=True,
    prompt_samples=3,
    drop_island_iter=20,
    prompt_decay=1.5,
    prompt_importance=5.0,
    max_islands=2,
    output_dir=cache_dir_2,
    history_name='repo_root_test',
    TIME_LIMIT=20,
    task_description=task_description,
    debug=False,
    min_improvement=0.01,
    n_jobs=1,
    COST_LIMIT=5,
)
Path('repo.txt').exists()